# 01 — Data Pipeline

## Task 1.2 — Open-Meteo weather fetch

Fetches hourly `temperature_2m` and `shortwave_radiation` (GHI) from the
Open-Meteo historical archive for a given location and date range.

In [ ]:
import ssl
import requests
import pandas as pd
from requests.adapters import HTTPAdapter

In [ ]:
class _WinCertAdapter(HTTPAdapter):
    """
    Mounts Windows system CA store so requests works behind corporate proxies.
    """
    def init_poolmanager(self, *args, **kwargs):
        ctx = ssl.create_default_context()
        ctx.load_default_certs(ssl.Purpose.SERVER_AUTH)
        kwargs["ssl_context"] = ctx
        super().init_poolmanager(*args, **kwargs)

_session = requests.Session()
_session.mount("https://", _WinCertAdapter())


def fetch_weather(lat, lon, start_date, end_date, timezone) -> pd.DataFrame:
    """
    Returns hourly UTC-aware DataFrame indexed by timestamp,
       with columns: temperature_2m, shortwave_radiation (Global Horizontal Irradiance, W/m²).

    Data is fetched and stored in UTC regardless of the timezone arg.
    Convert to local time only for plotting — solar physics works in UTC + longitude.
    """
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "hourly": "temperature_2m,shortwave_radiation",
        "timezone": "UTC",  # always fetch UTC — avoids DST NonExistentTime/AmbiguousTime errors
    }
    resp = _session.get(
        "https://archive-api.open-meteo.com/v1/archive",
        params=params,
        timeout=30,
    )
    resp.raise_for_status()

    hourly = resp.json()["hourly"]
    df = pd.DataFrame({
        "temperature_2m":      hourly["temperature_2m"],
        # shortwave_radiation == GHI (Global Horizontal Irradiance, W/m²)
        # pvlib will decompose this into DNI + DHI components
        "shortwave_radiation": hourly["shortwave_radiation"],
    }, index=pd.to_datetime(hourly["time"], utc=True))  # tz-aware UTC in one step, no tz_localize
    df.index.name = "timestamp"
    return df

In [ ]:
# Austin, TX — single source of truth; Week 2 pvlib uses these same coords
AUSTIN_LAT = 30.2672
AUSTIN_LON  = -97.7431
LOCAL_TZ    = "America/Chicago"

In [ ]:
# Smoke test — full year to exercise DST transitions
df_weather = fetch_weather(
    lat=AUSTIN_LAT,
    lon=AUSTIN_LON,
    start_date="2018-01-01",
    end_date="2018-12-31",
    timezone=LOCAL_TZ,
)

assert df_weather.index.tz is not None,               "Index must be tz-aware"
assert not df_weather.index.has_duplicates,            "Duplicate timestamps (DST fold?)"
assert df_weather.index.is_monotonic_increasing,       "Index not sorted"

# Open-Meteo occasionally returns null for shortwave_radiation at range boundaries
n_nans = df_weather.isna().sum().sum()
if n_nans:
    print(f"WARNING: {n_nans} NaNs present — investigate")

print(f"Rows: {len(df_weather)} (expect 8760 for a non-leap year)")
print(f"Index dtype: {df_weather.index.dtype}")
print(df_weather.head())

## Task 1.3 — Merge skeleton (built and proven against a mock)

In [ ]:
import numpy as np

# ---------------------------------------------------------------------------
# Mock weather: 48 h of hourly UTC data (2018-01-01 – 2018-01-02)
# ---------------------------------------------------------------------------
_mock_weather_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_weather_df = pd.DataFrame({
    "temperature_2m":      np.linspace(5, 15, 48),
    "shortwave_radiation": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 400, 0, None),
}, index=_mock_weather_idx)
mock_weather_df.index.name = "timestamp"

# ---------------------------------------------------------------------------
# Mock energy: same window but with three deliberate defects:
#   (a) missing hour  — 2018-01-01 15:00 UTC dropped entirely
#   (b) NaN value     — household_load_kwh at 2018-01-01 20:00 set to NaN
#   (c) out-of-range  — two rows outside the weather window (before and after)
# ---------------------------------------------------------------------------
_energy_idx = pd.date_range("2018-01-01", periods=48, freq="h", tz="UTC")
mock_energy_df = pd.DataFrame({
    "household_load_kwh": np.random.default_rng(42).uniform(0.3, 1.2, 48),
    "actual_pv_yield_kwh": np.clip(np.sin(np.linspace(0, 4 * np.pi, 48)) * 2, 0, None),
}, index=_energy_idx)
mock_energy_df.index.name = "timestamp"

# (a) drop one hour
mock_energy_df = mock_energy_df.drop(pd.Timestamp("2018-01-01 15:00", tz="UTC"))

# (b) inject a NaN
mock_energy_df.loc[pd.Timestamp("2018-01-01 20:00", tz="UTC"), "household_load_kwh"] = np.nan

# (c) append two out-of-range rows
_extra = pd.DataFrame({
    "household_load_kwh":  [0.9, 0.8],
    "actual_pv_yield_kwh": [0.0, 0.0],
}, index=pd.to_datetime(["2017-12-31 23:00", "2018-01-03 00:00"], utc=True))
_extra.index.name = "timestamp"
mock_energy_df = pd.concat([mock_energy_df, _extra]).sort_index()

print(f"mock_weather rows : {len(mock_weather_df)}")
print(f"mock_energy rows  : {len(mock_energy_df)}  (48 base − 1 dropped + 2 out-of-range)")
print(f"Missing hour present in energy: {pd.Timestamp('2018-01-01 15:00', tz='UTC') not in mock_energy_df.index}")

In [ ]:
def build_unified_frame(weather_df: pd.DataFrame, energy_df: pd.DataFrame) -> pd.DataFrame:
    """
    Align energy data onto weather data on a complete hourly UTC index.
    Returns one frame: timestamp index +
      [temperature_2m, shortwave_radiation, household_load_kwh, actual_pv_yield_kwh]

    Join direction: weather range is canonical.
      - Week 2 physics prediction requires a weather row for every output timestamp;
        we cannot produce a solar estimate without it, so the weather window drives the index.
      - Energy rows outside that window are irrelevant and dropped by reindex.
      - Week 3 training handles residual NaNs in actual_pv_yield_kwh by dropping those rows
        at fit-time rather than imputing solar yield we didn't measure.

    Missing energy hours:
      - household_load_kwh  → linear interpolation. Load profile is smooth; a single missing
        hour between two known values is well-estimated. Forward-fill would copy stale data;
        dropping would lose the row for Week 3 training.
      - actual_pv_yield_kwh → left as NaN. Solar yield depends on actual cloud cover during
        that hour; interpolating it would fabricate training signal.
    """
    # 1. Canonical index: every hour in the weather window, no gaps
    canonical = pd.date_range(
        start=weather_df.index.min(),
        end=weather_df.index.max(),
        freq="h",
        tz="UTC",
    )

    # 2. Reindex both sources — out-of-range energy rows silently disappear, gaps become NaN
    weather_aligned = weather_df.reindex(canonical)
    energy_aligned  = energy_df.reindex(canonical)

    # 3. Impute energy columns per stated policy
    energy_aligned["household_load_kwh"] = (
        energy_aligned["household_load_kwh"].interpolate(method="time")
    )
    # actual_pv_yield_kwh: intentionally left as NaN where missing

    # 4. Combine and name index
    result = pd.concat([weather_aligned, energy_aligned], axis=1)
    result.index.name = "timestamp"
    return result

In [ ]:
result = build_unified_frame(mock_weather_df, mock_energy_df)

# Index integrity
assert result.index.tz is not None,                  "Index must be tz-aware"
assert not result.index.has_duplicates,               "Duplicate timestamps"
assert result.index.is_monotonic_increasing,          "Index not sorted"

# Canonical index — same length as weather, no gaps
assert len(result) == len(mock_weather_df),           "Row count must match weather window"

# (c) Out-of-range energy rows must be gone
assert pd.Timestamp("2017-12-31 23:00", tz="UTC") not in result.index, "Pre-range row leaked in"
assert pd.Timestamp("2018-01-03 00:00", tz="UTC") not in result.index, "Post-range row leaked in"

# (a) Missing hour must now exist (filled by interpolation for load, NaN for PV)
missing_ts = pd.Timestamp("2018-01-01 15:00", tz="UTC")
assert missing_ts in result.index,                    "Missing hour not restored by reindex"
assert pd.notna(result.loc[missing_ts, "household_load_kwh"]), "load not interpolated at missing hour"
assert pd.isna(result.loc[missing_ts, "actual_pv_yield_kwh"]), "PV yield should stay NaN at missing hour"

# (b) Explicit NaN in load must also be interpolated
nan_ts = pd.Timestamp("2018-01-01 20:00", tz="UTC")
assert pd.notna(result.loc[nan_ts, "household_load_kwh"]), "load NaN not interpolated"

print(result.info())
print(f"\nNaNs per column:\n{result.isna().sum()}")